In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets pillow

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (fix this in Settings!)")

In [ ]:
import pandas as pd

projections = pd.read_csv("/kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_projections.csv")
reports = pd.read_csv("/kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_reports.csv")

print(projections.columns.tolist())
print(projections.head(3))
print()
print(reports.columns.tolist())
print(reports.head(3))

In [ ]:
print(reports.isna().sum())
print()
print(reports.iloc[0])

In [ ]:
reports['findings'].count()

In [ ]:
reports = reports.dropna(subset=["findings"]).copy()

final_df = pd.merge(reports, projections, on="uid", how="inner")

print(f"Final dataset shape: {final_df.shape}")
print(final_df.head())


In [ ]:
final_df['findings'] = final_df['findings'].str.replace("XXXX","",regex=False)
final_df['findings'] = final_df['findings'].str.replace(r"\s+"," ",regex=True).str.strip()
print(final_df[final_df['findings'].str.contains("XXXX")].shape[0])

In [ ]:
uids = final_df['uid'].unique()

from sklearn.model_selection import train_test_split
train, other = train_test_split(uids,test_size=0.3, random_state=42)
val, test = train_test_split(other, test_size = 0.5, random_state=42)

train_ds = final_df[final_df['uid'].isin(train)]
val_ds = final_df[final_df['uid'].isin(val)]
test_ds = final_df[final_df['uid'].isin(test)]


In [ ]:
train_ds['findings'].count()

In [ ]:
normal_counts = (train_ds['MeSH']=='normal').sum()
print(normal_counts)

In [ ]:
from collections import Counter
import re

count_list = []
mesh_series = train_ds[train_ds['MeSH']!='normal']['MeSH']
for value in mesh_series:
    count_list.extend(value.split(";"))
count_list = [value.strip().lower() for value in count_list]
count_list = [re.sub(r"\s+", " ", value) for value in count_list]
counts = Counter(count_list)
#for key, value in counts.items():
    #print(key, value)

In [ ]:
parents_counter = Counter()

for key, value in counts.items():
    parent = key.split('/')[0]
    parents_counter[parent] += value

#for key, value in parents_counter.items():
    #print(key, value)

In [ ]:
import math
weights = {} 
for key,value in parents_counter.items(): 
    weights[key] = 1/math.sqrt(value)
def assign_weight(row):

    if pd.isna(row['MeSH']):
        return None

    categories = row['MeSH'].split(";")

    categories = [value.lower().strip() for value in categories]

    categories = [
        re.sub(r'\s+', " ", category)
        for category in categories
    ]

    weight = []

    for category in categories:
        category = category.split("/")[0]

        if category == "normal":
            weight.append(1 / math.sqrt(normal_counts))

        elif category in weights:
            weight.append(weights[category])

    if len(weight) == 0:
        return None

    return max(weight)

In [ ]:
train_ds['weight'] = train_ds.apply(assign_weight, axis=1)

In [ ]:
train_ds['weight'].isna().sum()

In [ ]:
surviving_uids = reports["uid"]
expected_rows = projections[projections["uid"].isin(surviving_uids)].shape[0]
print("Expected:", expected_rows, "| Actual:", final_df.shape[0])

In [ ]:
print(set(train) & set(val),
set(train) & set(test),
set(val) & set(test))

print(len(train_ds), len(test_ds), len(val_ds))
print(train_ds.shape, test_ds.shape, val_ds.shape)

In [ ]:
val_ds['weight'] = 1.0

In [ ]:
pip install --upgrade torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu128

In [ ]:
!pip uninstall -y torchaudio

In [ ]:
from transformers import PaliGemmaProcessor

processor = PaliGemmaProcessor.from_pretrained(
    "google/paligemma-3b-pt-224",
     token="YOUR_HF_TOKEN_HERE"
)

In [ ]:
from PIL import Image
import os
import torch
from torch.utils.data import Dataset

class VQADataset(Dataset):
    def __init__(self, df, processor, image_directory):
        self.df = df
        self.processor = processor
        self.image_directory = image_directory

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_name = row["filename"]
        image_path = os.path.join(self.image_directory, image_name)
        image = Image.open(image_path).convert("RGB")
        prompt = "<image> Describe the findings in this chest X-ray."
        report_text = row["findings"]

        inputs = self.processor(
            images=image,
            text=prompt,
            suffix=report_text,
            return_tensors="pt"
        )

        inputs = {
            key: value.squeeze(0) for key, value in inputs.items()
        }

        inputs["weight"] = torch.tensor(row["weight"], dtype=torch.float)

        return inputs

In [ ]:
!pip uninstall -y torchao

In [ ]:
import torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import LoraConfig,get_peft_model

processor = AutoProcessor.from_pretrained("google/paligemma-3b-pt-224"
                                         ,token="YOUR_HF_TOKEN_HERE")
model = PaliGemmaForConditionalGeneration.from_pretrained(
    "google/paligemma-3b-pt-224",
    token="YOUR_HF_TOKEN_HERE",
    torch_dtype = torch.bfloat16,
    device_map = {"":0},
)

lora_config = LoraConfig(
    r = 8,
    lora_alpha = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
image_directory = "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized"
train_dataset = VQADataset(train_ds.reset_index(drop=True), processor, image_directory)
sample = train_dataset[0]
for k, v in sample.items():
    print(k, v.shape, v.dtype)

In [ ]:
small_ds = train_ds.head(8)

small_pt_dataset = VQADataset(
    df = small_ds,
    processor = processor,
    image_directory = image_directory
)

In [ ]:
sample = small_pt_dataset[0]
for key,value in sample.items():
    print(key,value.shape)

In [ ]:
def collate_fn(examples):
    batch = {
        "input_ids": [example["input_ids"] for example in examples],
        "token_type_ids": [example["token_type_ids"] for example in examples],
        "attention_mask": [example["attention_mask"] for example in examples],
    }

    labels = [example["labels"] for example in examples]
    maximum_length = max(len(label) for label in labels)

    padded_labels = []

    for label in labels:
        pad_needed = maximum_length - len(label)

        if pad_needed > 0:
            padding = torch.full(
                (pad_needed,),
                -100,
                dtype=label.dtype
            )
            label = torch.cat([label, padding])

        padded_labels.append(label)

    batch = processor.tokenizer.pad(
        batch,
        padding=True,
        return_tensors="pt"
    )

    batch["labels"] = torch.stack(padded_labels)

    batch["pixel_values"] = torch.stack(
        [example["pixel_values"] for example in examples]
    )

    batch["weight"] = torch.stack(
        [example["weight"] for example in examples]
    )

    return batch

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import EarlyStoppingCallback

image_directory = "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized"
train_dataset = VQADataset(train_ds.reset_index(drop=True), processor, image_directory)
val_dataset = VQADataset(val_ds.reset_index(drop=True), processor, image_directory)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        weights = inputs.pop("weight")
        outputs = model(**inputs)
        logits = outputs.logits
        labels = inputs["labels"]

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = torch.nn.CrossEntropyLoss(reduction="none", ignore_index=-100)
        token_losses = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1)
        )
        token_losses = token_losses.view(shift_labels.size())

        per_example_loss = token_losses.sum(dim=1) / (shift_labels != -100).sum(dim=1).clamp(min=1)
        weighted_loss = (per_example_loss * weights).mean()

        return (weighted_loss, outputs) if return_outputs else weighted_loss


training_args = TrainingArguments(
    output_dir="/kaggle/working/vlm-medical-lora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=6,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    dataloader_num_workers=4,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train(resume_from_checkpoint="/kaggle/input/models/dreams971/checkpoint-2264/transformers/default/1/checkpoint-2264")

In [ ]:
print(trainer.state.best_metric)
print(trainer.state.best_model_checkpoint)

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/checkpoint-2264-backup", "zip", "/kaggle/working/vlm-medical-lora/checkpoint-2264")

In [ ]:
save_path = "/kaggle/working/vlm-medical-lora-final_balanced_10epoch"

trainer.save_model(save_path)
processor.save_pretrained(save_path)

print(f"Model saved to: {save_path}")

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/vlm-medical-lora-final_balanced_10epoch",
    "zip",
    "/kaggle/working/vlm-medical-lora-final_balanced_10epoch"
)

In [ ]:
print(processor.tokenizer.padding_side)

In [ ]:
batch = collate_fn([small_pt_dataset[0], small_pt_dataset[3]])
print(batch["labels"])

In [ ]:
import os
import zipfile
import torch

from transformers import PaliGemmaForConditionalGeneration, PaliGemmaProcessor, AutoProcessor
from peft import PeftModel

model_path = "/kaggle/input/models/dreams971/final-/transformers/default/1/checkpoint-2264"


base_model_id = "google/paligemma-3b-pt-224"

processor = AutoProcessor.from_pretrained("google/paligemma-3b-pt-224"
                                         ,token="YOUR_HF_TOKEN_HERE")

model = PaliGemmaForConditionalGeneration.from_pretrained(
    base_model_id,
    token="YOUR_HF_TOKEN_HERE",
    torch_dtype=torch.float16
).to("cuda")

model = PeftModel.from_pretrained(
    model,
    model_path
)

model.eval()


In [ ]:
from PIL import Image
image_directory = "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized"
for idx in [0, 5, 10, 15]:
    row = test_ds.iloc[idx]
    image_path = os.path.join(image_directory, row["filename"])
    image = Image.open(image_path).convert("RGB")
    prompt = "<image> Describe the findings in this chest X-ray."
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output_ids = model.generate(
        **inputs, 
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.2
        )
    
    input_len = inputs["input_ids"].shape[-1]
    generated_text = processor.decode(output_ids[0][input_len:], skip_special_tokens=True)
    print(row["filename"], "->", generated_text)
    print("ACTUAL:", row["findings"])
    print()

In [ ]:
print(train_ds['findings'].str.contains("clear", case=False).mean())
print(train_ds['findings'].str.contains("no effusion|no pneumothorax|no acute", case=False).mean())

In [ ]:
token_lengths = train_ds['findings'].apply(lambda x: len(processor.tokenizer(x)['input_ids']))
print(token_lengths.describe())
print((token_lengths > 256).mean())

In [ ]:
print(train_ds[token_lengths > 256]['findings'].str.contains("no effusion|no pneumothorax|no acute", case=False).mean())
print(train_ds[token_lengths <= 256]['findings'].str.contains("no effusion|no pneumothorax|no acute", case=False).mean())

In [ ]:
!pip install -q evaluate rouge_score bert-score

In [ ]:
import torch
from tqdm import tqdm
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from peft import PeftModel
from evaluate import load
from PIL import Image

# 1. Load Evaluation Metrics
rouge = load("rouge")
bertscore = load("bertscore")

# 2. Setup Model and Processor
base_model_id = "google/paligemma-3b-pt-224"
processor = AutoProcessor.from_pretrained(base_model_id, token="YOUR_HF_TOKEN_HERE")

model = PaliGemmaForConditionalGeneration.from_pretrained(
    base_model_id,
    token="YOUR_HF_TOKEN_HERE",
    torch_dtype=torch.float16
).to("cuda")

# Load your final trained LoRA adapter path
model_path = "/kaggle/input/models/dreams971/final-/transformers/default/1/checkpoint-2264"
model = PeftModel.from_pretrained(model, model_path)
model.eval()

# 3. Generate Predictions on Test Set
predictions = []
references = []

image_directory = "/kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized"

print("Generating predictions on test set...")
for idx in tqdm(range(len(test_ds))):
    row = test_ds.iloc[idx]
    image_path = os.path.join(image_directory, row["filename"])
    image = Image.open(image_path).convert("RGB")
    
    prompt = "<image> Describe the findings in this chest X-ray."
    inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    
    with torch.no_log_grad() if hasattr(torch, "no_log_grad") else torch.no_grad():
        output_ids = model.generate(
            **inputs, 
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2
        )
    
    input_len = inputs["input_ids"].shape[-1]
    generated_text = processor.decode(output_ids[0][input_len:], skip_special_tokens=True)
    
    predictions.append(generated_text)
    references.append(row["findings"])

# 4. Compute ROUGE Scores
rouge_results = rouge.compute(predictions=predictions, references=references)
print("\n--- ROUGE Scores ---")
for key, value in rouge_results.items():
    print(f"{key}: {value:.4f}")

# 5. Compute BERTScore (Semantic Similarity)
print("\nComputing BERTScore...")
bert_results = bertscore.compute(predictions=predictions, references=references, lang="en", model_type="distilbert-base-uncased")
print(f"BERTScore Precision: {sum(bert_results['precision'])/len(bert_results['precision']):.4f}")
print(f"BERTScore Recall:    {sum(bert_results['recall'])/len(bert_results['recall']):.4f}")
print(f"BERTScore F1:        {sum(bert_results['f1'])/len(bert_results['f1']):.4f}")

In [ ]:
import pandas as pd

# Assuming test_ds has a 'MeSH' column
normal_mask = test_ds['MeSH'] == 'normal'

# Filter predictions and references for abnormal cases only
abnormal_preds = [p for p, is_norm in zip(predictions, normal_mask) if not is_norm]
abnormal_refs = [r for r, is_norm in zip(references, normal_mask) if not is_norm]

print(f"Evaluating on {len(abnormal_preds)} abnormal/pathology test cases...")

# Compute BERTScore specifically for abnormal cases
abnormal_bert = bertscore.compute(predictions=abnormal_preds, references=abnormal_refs, lang="en", model_type="distilbert-base-uncased")
print(f"Abnormal Cases BERTScore F1: {sum(abnormal_bert['f1'])/len(abnormal_bert['f1']):.4f}")

In [2]:
# CELL 1 — install (run once per session)
!pip install -q -U streamlit transformers accelerate peft
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared
print("✅ Installed")

✅ Installed


In [3]:
!pip uninstall -y torchao

In [4]:
%%writefile /kaggle/working/app.py
# =====================================================================
#  RadioLens — Chest X-ray Report Studio  (Streamlit)
#  PaliGemma-3B + LoRA adapter  ->  streamed findings report
# =====================================================================
import io
import os
import re
import gc
import time
import html
import base64
import traceback
from threading import Thread

import streamlit as st
import torch
from PIL import Image

# ----------------------------- Config -----------------------------
DEFAULT_ADAPTER = os.environ.get("VLM_MODEL_PATH", "/kaggle/input/models/dreams971/-/transformers/default/1")
DEFAULT_BASE = os.environ.get("VLM_BASE_MODEL", "google/paligemma-3b-pt-224")
HF_TOKEN = os.environ.get("HF_TOKEN") or None
DEFAULT_PROMPT = "Describe the findings in this chest X-ray."

CUDA = torch.cuda.is_available()
DEVICE_LABEL = torch.cuda.get_device_name(0) if CUDA else "CPU (slow)"

st.set_page_config(
    page_title="RadioLens · Chest X-ray Report Studio",
    page_icon="🩻",
    layout="wide",
    initial_sidebar_state="expanded",
)

# ----------------------------- Styling -----------------------------
CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&family=JetBrains+Mono:wght@400;500;600&display=swap');

:root{
  --bg:#060a12; --panel:rgba(14,22,38,.72); --panel-2:#0b1322;
  --line:rgba(125,211,252,.14); --line-strong:rgba(34,211,238,.45);
  --txt:#e6edf7; --muted:#8a9bb5;
  --cyan:#22d3ee; --violet:#8b5cf6; --green:#34d399; --amber:#fbbf24; --red:#f87171;
  --mono:'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;
}

html, body, .stApp, .stMarkdown, p, label, input, textarea, button{
  font-family:'Inter',system-ui,-apple-system,'Segoe UI',Roboto,sans-serif;
}

/* ---------- Background: glow + engineering grid ---------- */
.stApp{
  background:
    radial-gradient(1100px 560px at 8% -8%, rgba(34,211,238,.14), transparent 60%),
    radial-gradient(900px 520px at 105% 0%, rgba(139,92,246,.15), transparent 60%),
    radial-gradient(800px 500px at 50% 125%, rgba(34,211,238,.07), transparent 60%),
    linear-gradient(rgba(125,211,252,.03) 1px, transparent 1px) 0 0/48px 48px,
    linear-gradient(90deg, rgba(125,211,252,.03) 1px, transparent 1px) 0 0/48px 48px,
    var(--bg);
  background-attachment: fixed;
  color:var(--txt);
}
header[data-testid="stHeader"]{background:transparent;}
#MainMenu, footer, [data-testid="stDeployButton"]{visibility:hidden;}
.block-container{max-width:1240px; padding-top:2.2rem; padding-bottom:4rem;}

/* ---------- Sidebar ---------- */
section[data-testid="stSidebar"]{
  background:linear-gradient(180deg,#0a1224,#070c18);
  border-right:1px solid var(--line);
}
.side-brand{display:flex;align-items:center;gap:.6rem;font-weight:700;margin-bottom:.4rem}
.side-h{font-family:var(--mono);font-size:.68rem;letter-spacing:.2em;text-transform:uppercase;
  color:var(--cyan);margin:1.2rem 0 .4rem 0;padding-bottom:.35rem;border-bottom:1px solid var(--line)}

/* ---------- Hero ---------- */
.brand{display:flex;align-items:center;gap:.7rem;margin-bottom:1rem}
.brand-mark{width:42px;height:42px;border-radius:13px;display:grid;place-items:center;
  background:linear-gradient(135deg,rgba(34,211,238,.25),rgba(139,92,246,.25));
  border:1px solid var(--line-strong);box-shadow:0 0 28px rgba(34,211,238,.28)}
.brand-name{font-weight:700;letter-spacing:.02em;font-size:1.05rem}
.brand-name b{color:var(--cyan)}
.brand-ver{font-family:var(--mono);font-size:.68rem;color:var(--muted);border:1px solid var(--line);
  border-radius:6px;padding:.1rem .4rem;margin-left:.2rem}
.title{font-size:clamp(2rem,4.4vw,3.3rem);line-height:1.05;font-weight:800;letter-spacing:-.03em;margin:0 0 .7rem 0}
.grad{background:linear-gradient(90deg,#67e8f9,#22d3ee 35%,#a78bfa);-webkit-background-clip:text;
  background-clip:text;color:transparent}
.lead{color:var(--muted);max-width:740px;font-size:1.02rem;line-height:1.65;margin:0 0 1.1rem 0}
.pills{display:flex;flex-wrap:wrap;gap:.5rem;margin-bottom:1.6rem}
.pill{display:inline-flex;align-items:center;gap:.5rem;padding:.38rem .8rem;border-radius:999px;
  font-family:var(--mono);font-size:.74rem;color:#b8c7de;background:rgba(255,255,255,.035);
  border:1px solid var(--line)}
.dot{width:8px;height:8px;border-radius:50%;background:var(--amber);box-shadow:0 0 10px var(--amber);display:inline-block}
.dot.on{background:var(--green);box-shadow:0 0 10px var(--green);animation:pulse 2s infinite}
@keyframes pulse{0%,100%{opacity:1}50%{opacity:.4}}

/* ---------- Cards ---------- */
.st-key-input_card, .st-key-output_card{
  background:var(--panel);border:1px solid var(--line);border-radius:22px;padding:1.4rem 1.4rem 1.2rem;
  backdrop-filter:blur(14px);-webkit-backdrop-filter:blur(14px);
  box-shadow:0 24px 70px rgba(0,0,0,.4), inset 0 1px 0 rgba(255,255,255,.05);
}
.sec{display:flex;align-items:center;justify-content:space-between;margin:0 0 .6rem 0}
.sec-title{display:flex;align-items:center;gap:.65rem;font-weight:700;font-size:1.08rem}
.sec-num{font-family:var(--mono);font-size:.7rem;color:var(--cyan);border:1px solid var(--line-strong);
  border-radius:6px;padding:.12rem .42rem;background:rgba(34,211,238,.08)}
.sec-sub{color:var(--muted);font-size:.76rem;font-family:var(--mono)}

/* ---------- Widgets ---------- */
label[data-testid="stWidgetLabel"] p{font-family:var(--mono);font-size:.7rem;text-transform:uppercase;
  letter-spacing:.14em;color:var(--muted)}
[data-testid="stFileUploaderDropzone"]{
  background:rgba(34,211,238,.035);border:1.5px dashed rgba(34,211,238,.38);border-radius:16px;transition:all .25s}
[data-testid="stFileUploaderDropzone"]:hover{
  background:rgba(34,211,238,.08);border-color:var(--cyan);box-shadow:0 0 34px rgba(34,211,238,.14)}
[data-testid="stFileUploaderDropzone"] button{border-radius:10px;border:1px solid var(--line-strong);
  background:rgba(34,211,238,.08)}
div[data-baseweb="textarea"], div[data-baseweb="input"], div[data-baseweb="select"] > div{
  background:var(--panel-2)!important;border:1px solid var(--line)!important;border-radius:12px!important}
div[data-baseweb="textarea"]:focus-within, div[data-baseweb="input"]:focus-within{
  border-color:var(--cyan)!important;box-shadow:0 0 0 3px rgba(34,211,238,.16)!important}
div[data-baseweb="base-input"]{background:transparent!important}
.stTextArea textarea{font-family:var(--mono)!important;font-size:.88rem!important;background:transparent!important;color:#cfe9f5!important}

.stButton>button, .stDownloadButton>button{
  width:100%;border-radius:14px;font-weight:700;letter-spacing:.02em;padding:.72rem 1.2rem;transition:all .2s;
  border:1px solid var(--line-strong);background:rgba(34,211,238,.06);color:var(--txt)}
.stButton>button:hover, .stDownloadButton>button:hover{background:rgba(34,211,238,.13);border-color:var(--cyan);color:#fff}
.stButton>button[kind="primary"], .stButton>button[data-testid="stBaseButton-primary"]{
  background:linear-gradient(135deg,#06b6d4,#6366f1 60%,#8b5cf6);border:0;color:#fff;font-size:1rem;
  box-shadow:0 10px 32px rgba(34,211,238,.28), inset 0 0 0 1px rgba(255,255,255,.14)}
.stButton>button[kind="primary"]:hover, .stButton>button[data-testid="stBaseButton-primary"]:hover{
  transform:translateY(-2px);box-shadow:0 16px 44px rgba(99,102,241,.45), inset 0 0 0 1px rgba(255,255,255,.2)}
.stButton>button:disabled{opacity:.38;filter:grayscale(.7);box-shadow:none!important;transform:none!important}
[data-testid="stExpander"]{border:1px solid var(--line)!important;border-radius:14px!important;background:rgba(255,255,255,.02)}
[data-testid="stCaptionContainer"]{color:var(--muted)}

/* ---------- X-ray viewer ---------- */
.xr-frame{position:relative;border-radius:16px;overflow:hidden;background:#02050b;
  border:1px solid rgba(34,211,238,.28);box-shadow:0 0 44px rgba(34,211,238,.08), inset 0 0 60px rgba(0,0,0,.6);
  margin:.7rem 0 1rem 0}
.xr-frame img{display:block;width:100%;max-height:480px;object-fit:contain;transition:filter .4s}
.xr-frame.scanning img{filter:brightness(1.15) contrast(1.15) saturate(1.2)}
.xr-frame .c{position:absolute;width:22px;height:22px;border:2px solid var(--cyan);opacity:.9;filter:drop-shadow(0 0 6px var(--cyan))}
.c.tl{top:10px;left:10px;border-right:0;border-bottom:0;border-top-left-radius:6px}
.c.tr{top:10px;right:10px;border-left:0;border-bottom:0;border-top-right-radius:6px}
.c.bl{bottom:10px;left:10px;border-right:0;border-top:0;border-bottom-left-radius:6px}
.c.br{bottom:10px;right:10px;border-left:0;border-top:0;border-bottom-right-radius:6px}
.scan{position:absolute;left:0;right:0;height:90px;pointer-events:none;
  background:linear-gradient(to bottom,transparent,rgba(34,211,238,.32));animation:scan 2.1s ease-in-out infinite}
.scan::after{content:"";position:absolute;left:0;right:0;bottom:0;height:2px;background:var(--cyan);box-shadow:0 0 16px 2px var(--cyan)}
@keyframes scan{0%{top:-90px}100%{top:100%}}
.xr-tag{position:absolute;bottom:12px;font-family:var(--mono);font-size:.66rem;letter-spacing:.06em;
  color:#bfefff;background:rgba(2,8,18,.72);border:1px solid var(--line);border-radius:6px;padding:.18rem .5rem;
  max-width:55%;overflow:hidden;text-overflow:ellipsis;white-space:nowrap}
.xr-tag.l{left:40px}.xr-tag.r{right:40px}
.xr-frame.empty{min-height:210px;display:grid;place-items:center;border-style:dashed;background:rgba(2,5,11,.55)}
.xr-empty-txt{text-align:center;font-family:var(--mono);font-size:.78rem;letter-spacing:.2em;color:#4f6a86}
.xr-empty-txt span{display:block;margin-top:.4rem;letter-spacing:.04em;font-size:.72rem;color:#3d556f}

/* ---------- Report ---------- */
.report{border:1px solid var(--line);border-radius:16px;overflow:hidden;
  background:linear-gradient(180deg,rgba(11,19,34,.92),rgba(7,13,25,.92));margin-top:.5rem}
.report-head{display:flex;align-items:center;justify-content:space-between;gap:.6rem;padding:.72rem 1rem;
  border-bottom:1px solid var(--line);font-family:var(--mono);font-size:.7rem;letter-spacing:.16em;
  text-transform:uppercase;color:var(--muted)}
.report-head .l{display:flex;align-items:center;gap:.55rem}
.tag{padding:.16rem .5rem;border-radius:6px;border:1px solid rgba(251,191,36,.4);color:var(--amber);
  background:rgba(251,191,36,.07);letter-spacing:.08em;font-size:.64rem;white-space:nowrap}
.tag.err{border-color:rgba(248,113,113,.5);color:var(--red);background:rgba(248,113,113,.08)}
.report-body{padding:.6rem 1.2rem 1.1rem}
.line{display:flex;gap:.85rem;padding:.6rem 0;border-bottom:1px dashed rgba(125,211,252,.09)}
.line:last-child{border-bottom:0}
.ln{font-family:var(--mono);font-size:.7rem;color:var(--cyan);opacity:.75;min-width:1.7rem;padding-top:.3rem}
.lt{font-size:1.02rem;line-height:1.7;color:#dbe6f5}
.cursor{display:inline-block;width:8px;height:1.05em;background:var(--cyan);margin-left:3px;vertical-align:-.16em;
  animation:blink 1s steps(2) infinite;box-shadow:0 0 12px var(--cyan)}
@keyframes blink{50%{opacity:0}}
.sk{height:14px;border-radius:7px;margin:1rem 0;
  background:linear-gradient(90deg,rgba(255,255,255,.04),rgba(34,211,238,.2),rgba(255,255,255,.04));
  background-size:200% 100%;animation:shimmer 1.4s linear infinite}
@keyframes shimmer{0%{background-position:200% 0}100%{background-position:-200% 0}}
.errmsg{color:#fecaca;font-size:.95rem;line-height:1.6;padding:.9rem 0 .2rem}
.errhint{color:#94a9c4;font-size:.86rem;line-height:1.6;padding-bottom:.6rem}
.chips{display:flex;flex-wrap:wrap;gap:.5rem;margin:.9rem 0 .8rem}
.chip{font-family:var(--mono);font-size:.74rem;padding:.3rem .65rem;border-radius:8px;
  background:rgba(34,211,238,.06);border:1px solid var(--line);color:#a9dff0}
.chip b{color:#fff;font-weight:600}
.empty{border:1.5px dashed var(--line);border-radius:16px;padding:3.2rem 1.5rem;text-align:center;color:var(--muted);margin-top:.5rem}
.empty svg{opacity:.85;margin-bottom:.9rem}
.empty b{color:var(--txt);display:block;margin-bottom:.3rem;font-size:1.02rem}
.disclaimer{display:flex;gap:.7rem;align-items:flex-start;margin-top:1.6rem;padding:.9rem 1.1rem;border-radius:14px;
  border:1px solid rgba(251,191,36,.28);background:rgba(251,191,36,.05);color:#e7d29a;font-size:.85rem;line-height:1.55}
.foot{text-align:center;color:#51657f;font-family:var(--mono);font-size:.7rem;letter-spacing:.14em;margin-top:1.6rem}
</style>
"""


# ----------------------------- HTML helpers -----------------------------
def flat(s: str) -> str:
    """Collapse a snippet to a single line so Streamlit's markdown never breaks the HTML block."""
    return " ".join(line.strip() for line in s.strip().splitlines())


ICON_SCAN = (
    '<svg width="22" height="22" viewBox="0 0 24 24" fill="none" stroke="#67e8f9" stroke-width="1.8" '
    'stroke-linecap="round" stroke-linejoin="round"><path d="M4 8V6a2 2 0 0 1 2-2h2"/><path d="M16 4h2a2 2 0 0 1 2 2v2"/>'
    '<path d="M20 16v2a2 2 0 0 1-2 2h-2"/><path d="M8 20H6a2 2 0 0 1-2-2v-2"/><path d="M8 12h8"/></svg>'
)

HERO = flat(f"""
<div class="brand">
  <div class="brand-mark">{ICON_SCAN}</div>
  <div class="brand-name">Radio<b>Lens</b><span class="brand-ver">v1.0</span></div>
</div>
<div class="title">Chest X-ray <span class="grad">report studio</span></div>
<p class="lead">Upload a chest radiograph and a fine-tuned vision-language model drafts the findings for you,
streamed live, token by token.</p>
""")


def status_html(loaded: bool) -> str:
    dot = '<span class="dot on"></span>' if loaded else '<span class="dot"></span>'
    label = "Model ready" if loaded else "Model loads on first run"
    return flat(f"""
    <div class="pills">
      <span class="pill">{dot}{label}</span>
      <span class="pill">PaliGemma-3B · LoRA</span>
      <span class="pill">Indiana Univ. Chest X-rays</span>
      <span class="pill">{html.escape(DEVICE_LABEL)}</span>
    </div>""")


def sec_html(num: str, title: str, sub: str) -> str:
    return flat(f'<div class="sec"><div class="sec-title"><span class="sec-num">{num}</span>{title}</div>'
                f'<div class="sec-sub">{sub}</div></div>')


def frame_html(b64: str, name: str, size, scanning: bool) -> str:
    cls = "xr-frame scanning" if scanning else "xr-frame"
    scan = '<div class="scan"></div>' if scanning else ""
    state = "ANALYZING" if scanning else "READY"
    return flat(f"""
    <div class="{cls}">
      <img src="data:image/jpeg;base64,{b64}" alt="chest x-ray"/>
      <i class="c tl"></i><i class="c tr"></i><i class="c bl"></i><i class="c br"></i>
      {scan}
      <div class="xr-tag l">{html.escape(name)}</div>
      <div class="xr-tag r">{state} · {size[0]}×{size[1]}</div>
    </div>""")


VIEWER_EMPTY = flat("""
<div class="xr-frame empty">
  <i class="c tl"></i><i class="c tr"></i><i class="c bl"></i><i class="c br"></i>
  <div class="xr-empty-txt">NO IMAGE LOADED<span>upload a chest X-ray above</span></div>
</div>""")

EMPTY_REPORT = flat("""
<div class="empty">
  <svg width="54" height="54" viewBox="0 0 24 24" fill="none" stroke="#22d3ee" stroke-width="1.3"
       stroke-linecap="round" stroke-linejoin="round"><path d="M4 8V6a2 2 0 0 1 2-2h2"/><path d="M16 4h2a2 2 0 0 1 2 2v2"/>
       <path d="M20 16v2a2 2 0 0 1-2 2h-2"/><path d="M8 20H6a2 2 0 0 1-2-2v-2"/><path d="M8 12h8"/></svg>
  <b>Your report will appear here</b>
  Upload an X-ray on the left and press <b style="display:inline;color:#67e8f9">Generate report</b>.
</div>""")

DISCLAIMER = flat("""
<div class="disclaimer"><span>⚠️</span><span><b>Research prototype.</b> Reports are AI-generated from a model fine-tuned
on a small public dataset. This is not a medical device and must not be used for diagnosis or treatment decisions.
Do not upload identifiable patient data.</span></div>
<div class="foot">RADIOLENS · PALIGEMMA-3B + LORA · STREAMLIT</div>""")


def split_sentences(text: str):
    parts = re.split(r"(?<=[.!?])\s+", text.strip())
    return [p for p in parts if p]


def report_html(text: str, streaming: bool = False) -> str:
    sents = split_sentences(text) or [""]
    rows = []
    for i, s in enumerate(sents, 1):
        cursor = '<span class="cursor"></span>' if (streaming and i == len(sents)) else ""
        rows.append(f'<div class="line"><span class="ln">{i:02d}</span><span class="lt">{html.escape(s)}{cursor}</span></div>')
    label = "Generating findings" if streaming else "Findings"
    return flat(f"""
    <div class="report">
      <div class="report-head"><span class="l"><span class="dot on"></span>{label}</span><span class="tag">AI-generated · not for clinical use</span></div>
      <div class="report-body">{''.join(rows)}</div>
    </div>""")


def loading_html(msg: str) -> str:
    return flat(f"""
    <div class="report">
      <div class="report-head"><span class="l"><span class="dot"></span>{html.escape(msg)}</span><span class="tag">please wait</span></div>
      <div class="report-body">
        <div class="sk" style="width:92%"></div><div class="sk" style="width:78%"></div>
        <div class="sk" style="width:86%"></div><div class="sk" style="width:52%"></div>
      </div>
    </div>""")


def error_html(msg: str, hint: str) -> str:
    return flat(f"""
    <div class="report">
      <div class="report-head"><span class="l"><span class="dot" style="background:#f87171;box-shadow:0 0 10px #f87171"></span>Something went wrong</span><span class="tag err">error</span></div>
      <div class="report-body"><div class="errmsg">{html.escape(msg)}</div><div class="errhint">{html.escape(hint)}</div></div>
    </div>""")


def chips_html(h: dict) -> str:
    speed = h["tokens"] / h["secs"] if h["secs"] > 0 else 0.0
    items = [("⏱", f'{h["secs"]:.1f}s'), ("🔤", f'{h["tokens"]} tokens'), ("⚡", f"{speed:.1f} tok/s"), ("🎛", h["dtype"])]
    chips = "".join(f'<span class="chip">{i} <b>{html.escape(v)}</b></span>' for i, v in items)
    return flat(f'<div class="chips">{chips}</div>')


def hint_for(msg: str) -> str:
    m = msg.lower()
    if "adapter_config" in m or "no lora adapter" in m:
        return ("Check the 'LoRA adapter folder' in the sidebar. It must contain adapter_config.json and "
                "adapter_model.safetensors (the folder you saved with trainer.save_model).")
    if "401" in m or "gated" in m or "restricted" in m or "access to model" in m or "token" in m:
        return ("PaliGemma is a gated model. Add your Hugging Face token as a Kaggle Secret named HF_TOKEN "
                "(Add-ons → Secrets), make sure it is attached to this notebook, and re-run the launch cell.")
    if "out of memory" in m:
        return "GPU is full. Restart the notebook kernel (frees the training copy of the model), then re-run all cells."
    return "Open 'Technical details' below, or run the debug cell to see streamlit.log."


def to_b64(img: Image.Image, max_side: int = 900) -> str:
    im = img.copy()
    im.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    im.save(buf, format="JPEG", quality=88)
    return base64.b64encode(buf.getvalue()).decode()


# ----------------------------- Model -----------------------------
@st.cache_resource
def app_state():
    """Tiny process-wide dict so the status pill knows if the model is already in memory."""
    return {"loaded_key": None}


def find_adapter_dir(root: str):
    """Return the folder containing adapter_config.json (root itself or a sub-folder)."""
    if not root or not os.path.isdir(root):
        return None
    if os.path.isfile(os.path.join(root, "adapter_config.json")):
        return root
    found = []
    for dirpath, _, files in os.walk(root):
        if "adapter_config.json" in files:
            found.append(dirpath)
    if not found:
        return None
    final = [p for p in found if "checkpoint" not in p.lower()]
    return sorted(final or found)[-1]


def pick_dtype(precision: str):
    if not CUDA or precision == "float32":
        return torch.float32
    if precision == "float16":
        return torch.float16
    if precision == "bfloat16":
        return torch.bfloat16
    # auto: native bf16 on Ampere+ (capability >= 8), fp16 on T4 / P100
    return torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16


def _load_paligemma(path_or_id: str, dtype):
    from transformers import PaliGemmaForConditionalGeneration
    kw = {"token": HF_TOKEN}
    if CUDA:
        kw["device_map"] = {"": 0}
    try:
        model = PaliGemmaForConditionalGeneration.from_pretrained(path_or_id, dtype=dtype, **kw)
    except (TypeError, ValueError):  # older transformers only knows torch_dtype
        model = PaliGemmaForConditionalGeneration.from_pretrained(path_or_id, torch_dtype=dtype, **kw)
    if model.dtype != dtype:
        model = model.to(dtype)
    return model


@st.cache_resource(show_spinner=False)
def load_model(adapter_root: str, base_id: str, precision: str):
    from transformers import AutoProcessor
    dtype = pick_dtype(precision)
    adapter_dir = find_adapter_dir(adapter_root)

    if adapter_dir is not None:
        from peft import PeftModel
        base = _load_paligemma(base_id, dtype)
        model = PeftModel.from_pretrained(base, adapter_dir)
        model = model.merge_and_unload()  # merge LoRA into the weights -> faster inference
        proc_src = adapter_dir
    elif os.path.isfile(os.path.join(adapter_root, "config.json")):
        model = _load_paligemma(adapter_root, dtype)  # a fully merged model was saved instead
        proc_src = adapter_root
    else:
        raise FileNotFoundError(f"No LoRA adapter found (adapter_config.json missing) under: {adapter_root}")

    model.eval()
    try:
        processor = AutoProcessor.from_pretrained(proc_src)
    except Exception:
        processor = AutoProcessor.from_pretrained(base_id, token=HF_TOKEN)

    info = {"dtype": str(dtype).replace("torch.", ""), "adapter": adapter_dir or adapter_root, "base": base_id}
    return model, processor, info


def start_generation(model, processor, image, prompt, max_new_tokens, temperature, top_p, rep_pen):
    """Runs model.generate in a worker thread and returns a streamer to iterate over."""
    from transformers import TextIteratorStreamer

    text = prompt.strip() or DEFAULT_PROMPT
    if "<image>" not in text:  # same format as training: "<image> Describe the findings ..."
        text = "<image> " + text

    inputs = processor(images=image, text=text, return_tensors="pt")
    prepared = {}
    for k, v in inputs.items():
        if torch.is_tensor(v):
            v = v.to(model.device)
            if k == "pixel_values":
                v = v.to(model.dtype)
            prepared[k] = v

    streamer = TextIteratorStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True, timeout=600)
    gen_kwargs = dict(**prepared, streamer=streamer, max_new_tokens=int(max_new_tokens),
                      repetition_penalty=float(rep_pen))
    if temperature > 0:
        gen_kwargs.update(do_sample=True, temperature=float(temperature), top_p=float(top_p))
    else:
        gen_kwargs.update(do_sample=False)

    err = {}

    def _worker():
        try:
            with torch.inference_mode():
                model.generate(**gen_kwargs)
        except Exception as e:  # noqa: BLE001
            err["e"], err["tb"] = e, traceback.format_exc()
            streamer.end()  # unblock the iterator in the main thread

    thread = Thread(target=_worker, daemon=True)
    thread.start()
    return streamer, thread, err


# =====================================================================
#                                 UI
# =====================================================================
st.markdown(CSS.strip(), unsafe_allow_html=True)

state = app_state()
if "history" not in st.session_state:
    st.session_state.history = []

# ---------- Sidebar ----------
with st.sidebar:
    st.markdown(f'<div class="side-brand">{ICON_SCAN} Radio<span style="color:#22d3ee">Lens</span></div>',
                unsafe_allow_html=True)
    st.markdown('<div class="side-h">Model</div>', unsafe_allow_html=True)
    adapter_path = st.text_input("LoRA adapter folder", DEFAULT_ADAPTER)
    base_model = st.text_input("Base model", DEFAULT_BASE)
    precision = st.selectbox("Precision", ["auto", "float16", "bfloat16", "float32"], index=0)

    st.markdown('<div class="side-h">Generation</div>', unsafe_allow_html=True)
    max_new_tokens = st.slider("Max new tokens", 64, 512, 256, 32)
    temperature = st.slider("Temperature (0 = deterministic)", 0.0, 1.2, 0.0, 0.05)
    top_p = st.slider("Top-p", 0.5, 1.0, 0.9, 0.05, disabled=(temperature == 0))
    rep_pen = st.slider("Repetition penalty", 1.0, 1.5, 1.0, 0.05)

    st.markdown('<div class="side-h">Session</div>', unsafe_allow_html=True)
    if st.button("↻  Reload model"):
        load_model.clear()
        state["loaded_key"] = None
        gc.collect()
        if CUDA:
            torch.cuda.empty_cache()
        st.rerun()
    st.caption(f"Device: {DEVICE_LABEL}")

model_key = (adapter_path, base_model, precision)

# ---------- Hero ----------
st.markdown(HERO, unsafe_allow_html=True)
status_ph = st.empty()
status_ph.markdown(status_html(state.get("loaded_key") == model_key), unsafe_allow_html=True)

# ---------- Two-column studio ----------
left, right = st.columns([1, 1.08], gap="large")

image, b64 = None, None
with left:
    with st.container(key="input_card"):
        st.markdown(sec_html("01", "Input", "upload · prompt"), unsafe_allow_html=True)
        uploaded = st.file_uploader("Chest X-ray image", type=["png", "jpg", "jpeg", "bmp", "webp"])
        img_ph = st.empty()
        prompt = st.text_area("Prompt", DEFAULT_PROMPT, height=90)
        st.caption("The model was fine-tuned with this exact prompt, so keep it for the best results.")
        run = st.button("✦  Generate report", type="primary", disabled=uploaded is None)

        if uploaded is not None:
            try:
                image = Image.open(io.BytesIO(uploaded.getvalue())).convert("RGB")
                b64 = to_b64(image)
                img_ph.markdown(frame_html(b64, uploaded.name, image.size, False), unsafe_allow_html=True)
            except Exception as e:  # noqa: BLE001
                st.error(f"Could not read that image: {e}")
        else:
            img_ph.markdown(VIEWER_EMPTY, unsafe_allow_html=True)

with right:
    with st.container(key="output_card"):
        st.markdown(sec_html("02", "Findings report", "streamed live"), unsafe_allow_html=True)
        report_ph = st.empty()
        meta_ph = st.empty()
        failed = False

        if run and image is not None:
            try:
                img_ph.markdown(frame_html(b64, uploaded.name, image.size, True), unsafe_allow_html=True)
                already = state.get("loaded_key") == model_key
                report_ph.markdown(loading_html("Analyzing image" if already else "Loading model weights (first run only)"),
                                   unsafe_allow_html=True)

                model, processor, info = load_model(adapter_path, base_model, precision)
                state["loaded_key"] = model_key
                status_ph.markdown(status_html(True), unsafe_allow_html=True)
                report_ph.markdown(loading_html("Analyzing image"), unsafe_allow_html=True)

                t0 = time.time()
                streamer, thread, err = start_generation(model, processor, image, prompt,
                                                         max_new_tokens, temperature, top_p, rep_pen)
                text = ""
                for chunk in streamer:
                    text += chunk
                    report_ph.markdown(report_html(text, streaming=True), unsafe_allow_html=True)
                thread.join()
                if err:
                    raise err["e"]

                secs = time.time() - t0
                text = text.strip()
                n_tok = len(processor.tokenizer(text, add_special_tokens=False)["input_ids"]) if text else 0
                st.session_state.history.insert(0, {
                    "name": uploaded.name, "prompt": prompt, "report": text or "(empty output)",
                    "secs": secs, "tokens": n_tok, "dtype": info["dtype"], "time": time.strftime("%H:%M:%S"),
                })
            except Exception as e:  # noqa: BLE001
                failed = True
                msg = f"{type(e).__name__}: {e}"
                report_ph.markdown(error_html(msg[:400], hint_for(msg)), unsafe_allow_html=True)
                with st.expander("Technical details"):
                    st.code(traceback.format_exc(), language="text")
            finally:
                if image is not None:
                    img_ph.markdown(frame_html(b64, uploaded.name, image.size, False), unsafe_allow_html=True)

        if not failed:
            hist = st.session_state.history
            if hist:
                latest = hist[0]
                report_ph.markdown(report_html(latest["report"]), unsafe_allow_html=True)
                meta_ph.markdown(chips_html(latest), unsafe_allow_html=True)
                export = (f"Chest X-ray findings (AI-generated draft)\nFile: {latest['name']}\n"
                          f"Prompt: {latest['prompt']}\nGenerated: {latest['time']}\n\n{latest['report']}\n\n"
                          "-- Research prototype. Not a medical device. --\n")
                st.download_button("⬇  Download report (.txt)", export, file_name=f"report_{latest['name']}.txt",
                                   key=f"dl_{len(hist)}")
                with st.expander("Copy as plain text"):
                    try:
                        st.code(latest["report"], language="text", wrap_lines=True)
                    except TypeError:  # older Streamlit without wrap_lines
                        st.code(latest["report"], language="text")
            else:
                report_ph.markdown(EMPTY_REPORT, unsafe_allow_html=True)

# ---------- History ----------
if len(st.session_state.history) > 1:
    with st.expander(f"Session history · {len(st.session_state.history) - 1} earlier report(s)"):
        for h in st.session_state.history[1:]:
            st.caption(f"{h['name']} · {h['time']} · {h['secs']:.1f}s")
            st.markdown(report_html(h["report"]), unsafe_allow_html=True)

st.markdown(DISCLAIMER, unsafe_allow_html=True)


Overwriting /kaggle/working/app.py


In [5]:
# CELL 3 — launch the app + public tunnel
import os, re, time, subprocess, urllib.request
from IPython.display import display, HTML

# ---------------- CONFIG (edit these two) ----------------
ADAPTER_PATH = "/kaggle/input/models/dreams971/-/transformers/default/1"   # folder that contains adapter_config.json
BASE_MODEL   = "google/paligemma-3b-pt-224"
# ---------------------------------------------------------

# HF token comes from Kaggle Secrets (Add-ons -> Secrets -> label: HF_TOKEN). Never hard-code it in a cell.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("token")
    print("🔑 HF_TOKEN loaded from Kaggle Secrets")
except Exception as e:
    print("⚠️  No HF_TOKEN secret found. It is required to download the gated PaliGemma base weights.\n   ", e)

os.environ["VLM_MODEL_PATH"] = ADAPTER_PATH
os.environ["VLM_BASE_MODEL"] = BASE_MODEL

if not os.path.isdir(ADAPTER_PATH):
    print(f"⚠️  ADAPTER_PATH does not exist yet: {ADAPTER_PATH}\n    (you can still fix it later in the app's sidebar)")

# stop anything from a previous run ([s] trick stops pkill from matching its own shell)
subprocess.run('pkill -f "[s]treamlit run"; pkill -f "[c]loudflared"', shell=True)
time.sleep(2)

# 1) start Streamlit (dark theme flags match the custom CSS)
st_cmd = [
    "streamlit", "run", "/kaggle/working/app.py",
    "--server.port", "8501", "--server.headless", "true",
    "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
    "--browser.gatherUsageStats", "false",
    "--theme.base", "dark", "--theme.primaryColor", "#22d3ee",
    "--theme.backgroundColor", "#060a12", "--theme.secondaryBackgroundColor", "#0b1322",
    "--theme.textColor", "#e6edf7",
]
subprocess.Popen(st_cmd, stdout=open("/kaggle/working/streamlit.log", "w"), stderr=subprocess.STDOUT)

up = False
for _ in range(60):
    try:
        if urllib.request.urlopen("http://localhost:8501/_stcore/health", timeout=2).status == 200:
            up = True; break
    except Exception:
        time.sleep(1)
print("✅ Streamlit is up" if up else "❌ Streamlit did not start — run CELL 4 to see streamlit.log")

# 2) start the Cloudflare tunnel and grab the public URL
subprocess.Popen(["/kaggle/working/cloudflared", "tunnel", "--url", "http://localhost:8501", "--no-autoupdate"],
                 stdout=open("/kaggle/working/tunnel.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(45):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/kaggle/working/tunnel.log").read())
    if m:
        url = m.group(0); break

if url:
    display(HTML(f'<a href="{url}" target="_blank" style="font-size:20px;font-weight:700">🔗 Open RadioLens → {url}</a>'))
else:
    print("❌ Tunnel URL not found — is Internet ON? Run CELL 4 and check tunnel.log")

🔑 HF_TOKEN loaded from Kaggle Secrets
✅ Streamlit is up
